# 01 — Baseline (not-smart rule engine)

Control group: the regex/keyword heuristic from `main.py`, **without** the knowledge-base branch.

- Kernel: **Python 3.10+** (do not use `route-chatbot/.venv`, it is 3.9)
- No extra packages beyond `python-dotenv`
- Labels in `eval_queries.py` were written against this heuristic, so accuracy should be ~100%. That validates the loop, not that regex is a good router.


In [6]:
%pip install python-dotenv -q


Note: you may need to restart the kernel to use updated packages.


In [7]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))


poc_dir /Users/tushar/PravarAI/route-chatbot/poc
eval queries 16
ollama http://localhost:11434 llama3.1:8b | alt llama3.1:8b
openai model gpt-3.5-turbo | key set True
typesafe key set False
typesafe key set False


## `decide_route`


In [8]:
import re

GREETING_PATTERN = re.compile(
    r"^\s*(hi|hello|hey|good morning|good afternoon|good evening|thanks|thank you|bye|how are you)\b",
    re.IGNORECASE,
)
COMPLEX_KEYWORDS = (
    "code", "function", "debug", "algorithm", "write a", "explain why",
    "compare", "analyze", "design", "poem", "story", "essay", "strategy",
)


def decide_route(message: str) -> str:
    """Return \"ollama\" or \"openai\". No KB."""
    if GREETING_PATTERN.search(message):
        return "ollama"
    lower = message.lower()
    if any(keyword in lower for keyword in COMPLEX_KEYWORDS):
        return "openai"
    return "ollama"


## Eval (routing only)


In [9]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        result = decide_route(item["message"])
        if isinstance(result, tuple):
            predicted = result[0]
            extra = result[1] if len(result) > 1 else None
        else:
            predicted = result
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{extra}{err}")


accuracy 16/16 (100%)  mean latency 0.0 ms  errors 0

  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hi there
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hello
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  hey
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  good morning
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  thanks
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  how are you
  [OK  ]     0.0 ms  exp=openai  pred=openai  write a function to reverse a linked list
  [OK  ]     0.0 ms  exp=openai  pred=openai  compare merge sort and quick sort
  [OK  ]     0.0 ms  exp=openai  pred=openai  debug this python code
  [OK  ]     0.0 ms  exp=openai  pred=openai  analyze the time complexity of this algorithm
  [OK  ]     0.0 ms  exp=openai  pred=openai  write a poem about the ocean
  [OK  ]     0.0 ms  exp=openai  pred=openai  design a strategy for caching
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  tell me something interesting
  [OK  ]     0.0 ms  exp=ollama  pred=ollama  what did you do t

## Notes (fill during the experiment)

- Ease of build / understand / remove: this is a few regexes.
- Blind spots: paraphrases (`"greetings"`), mixed intent (`"hi, write a function"` — greeting wins because it is checked first), anything not in the keyword list.


In [10]:
GENERATE = True  # flip to True during the experiment, not the scaffold

def call_ollama(message: str, model: str | None = None) -> str:
    import json
    import urllib.request

    payload = json.dumps({
        "model": model or OLLAMA_MODEL,
        "prompt": message,
        "stream": False,
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL.rstrip('/')}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode()).get("response", "").strip()


def call_openai(message: str) -> str:
    import json
    import urllib.request

    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is missing")
    payload = json.dumps({
        "model": OPENAI_MODEL,
        "messages": [{"role": "user", "content": message}],
        "max_tokens": 64,
    }).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode())
        return data["choices"][0]["message"]["content"].strip()


if GENERATE:
    for item in EVAL_QUERIES[:]:
        decision = decide_route(item["message"])
        label = decision[0] if isinstance(decision, tuple) else decision
        fn = call_openai if label == "openai" else call_ollama
        print("---", item["message"], "->", label)
        print(fn(item["message"])[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to actually call models.")


--- hi there -> ollama
How's it going? Is there something I can help you with or would you like to chat?

--- hello -> ollama
Hello! How are you doing today? Is there something I can help you with or would you like to chat?

--- hey -> ollama
What's up? Is there something I can help you with or would you like to chat?

--- good morning -> ollama
Good morning! Hope you're having a great start to your day! How can I assist you today?

--- thanks -> ollama
You're welcome! Is there anything else I can help you with?

--- how are you -> ollama
I'm just a computer program, so I don't have feelings or emotions like humans do. I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?

--- write a function to reverse a linked list -> openai
Here is a Python function that reverses a linked list:

```python
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class LinkedList:
    def __init__(self)